# Альфа-Банк × МФТИ — «Отклик на кредитный оффер». Финальное решение

**Задача.** Бинарная классификация: вероятность согласия корпоративного клиента на оффер по
овердрафту (`target_value`). **Метрика — ROC-AUC.** **Итоговый результат: LB ≈ 0.776.**

**Архитектура решения (ранговый ансамбль 4 моделей):**
1. **Feature engineering** — спред ставок, коридор лимитов, отношения активности, флаги пропусков.
2. **OpenFE** — автоматическая генерация признаков (топ-30), давшая решающий прирост.
3. **Градиентный бустинг** — LightGBM + CatBoost + XGBoost (XGBoost дотюнен Optuna по OOF), на
   признаках (engineered + OpenFE).
4. **FT-Transformer** (rtdl, 5-сид ансамбль) — декоррелированная нейросеть на всех данных.
5. Подмешивание известных меток для 8 721 строки сабмита, лежащих в train.

**Ключевые методологические уроки проекта:**
- Валидация строго **по времени** (test — будущее); ориентир — устойчивый сигнал, а не одно окно.
- Тюнинг под единичный holdout переобучается → тюнили по **OOF** (XGBoost дал +0.13 LB).
- Ручной feature engineering и target-encoding **проваливались** из-за covariate shift
  (adversarial AUC train/test ≈ 0.9997); **OpenFE** нашёл shift-устойчивые признаки.
- Принципиально разные классы моделей (бустинги + FT-Transformer) дают лучший ансамбль;
  TabPFN/ResNet не помогли.

> Требуется GPU (FT-Transformer) и заметное CPU-время (OpenFE).

In [ ]:
import warnings, time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
import torch, torch.nn as nn
from sklearn.preprocessing import QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from openfe import OpenFE, transform
from rtdl_revisiting_models import FTTransformer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
DEV = "cuda" if torch.cuda.is_available() else "cpu"
ID, TARGET, DATE = "front_id", "target_value", "decision_day"
CAT_COLS = ["db_group_last", "fl_adminarea"]
HOLDOUT_START, DATASET_START = "2025-03-01", "2024-02-01"
print("device:", DEV)

## 1. Загрузка данных

In [ ]:
train = pd.read_csv("train_apps.csv", parse_dates=[DATE])
test = pd.read_csv("test_apps.csv", parse_dates=[DATE])
sample = pd.read_csv("sample_submission.csv")
print("train:", train.shape, "| test:", test.shape, "| sample:", sample.shape)
print(f"Доля положительного класса: {train[TARGET].mean():.4f}")
print("Период train:", train[DATE].min().date(), "→", train[DATE].max().date())
print("Период test: ", test[DATE].min().date(), "→", test[DATE].max().date())

## 2. EDA: анализ взаимосвязей
Класс несбалансирован (~6 %); test — будущий период (covariate shift); пропуски структурированы.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
train[TARGET].value_counts().plot.bar(ax=ax[0], color=["#c0392b", "#27ae60"])
ax[0].set_title(f"Баланс классов (доля 1 = {train[TARGET].mean():.1%})")
train.set_index(DATE).resample("W").size().plot(ax=ax[1], label="train")
test.set_index(DATE).resample("W").size().plot(ax=ax[1], label="test")
ax[1].axvline(pd.Timestamp("2025-06-05"), color="k", ls="--", lw=1)
ax[1].set_title("Заявки по неделям: train → test (будущее)"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
na = train.isna().mean().sort_values(ascending=False); na = na[na > 0]
plt.figure(figsize=(9, 5)); na.plot.barh(color="#2980b9"); plt.gca().invert_yaxis()
plt.title("Доля пропусков по признакам"); plt.xlabel("доля NaN"); plt.tight_layout(); plt.show()

sub_ids, test_ids, train_ids = set(sample[ID]), set(test[ID]), set(train[ID])
print("Сабмит:", len(sub_ids), "id =", len(sub_ids & test_ids), "из test +",
      len(sub_ids & train_ids), "из train (метки известны → подставим)")

## 3. Feature engineering

In [ ]:
def build_features(df):
    df = df.copy()
    df["rate_spread"] = df["offered_rate"] - df["cb_rate"]
    df["rate_ratio"] = df["offered_rate"] / df["cb_rate"].replace(0, np.nan)
    df["band_width"] = df["overdraft_limit_max"] - df["overdraft_limit_min"]
    df["loan_in_band_pos"] = (df["loan_amount_last"] - df["overdraft_limit_min"]) / df["band_width"].replace(0, np.nan)
    df["loan_to_max"] = df["loan_amount_last"] / df["overdraft_limit_max"].replace(0, np.nan)
    df["loan_to_min"] = df["loan_amount_last"] / df["overdraft_limit_min"].replace(0, np.nan)
    df["loan_within_band"] = ((df["loan_amount_last"] >= df["overdraft_limit_min"]) &
                              (df["loan_amount_last"] <= df["overdraft_limit_max"])).astype(int)
    df["deb_ul_30_90_ratio"] = df["sum_deb_ul_30"] / df["sum_deb_ul_90"].replace(0, np.nan)
    df["cnt_ul_ip_30_90_ratio"] = df["cnt_deb_ul_ip_30"] / df["cnt_deb_ul_ip_90"].replace(0, np.nan)
    dd = df[DATE]
    df["dd_month"], df["dd_quarter"] = dd.dt.month, dd.dt.quarter
    df["dd_dow"], df["dd_day"] = dd.dt.dayofweek, dd.dt.day
    df["dd_days_from_start"] = (dd - pd.Timestamp(DATASET_START)).dt.days
    na_flag = ["corp_credit_products", "sum_deb_ul_90", "sum_deb_ul_30", "loan_rev_max_start_non_fin",
               "loan_rev_min_start_fin", "overdraft_app_term_max_360", "days_from_authperson_registration",
               "sum_deb_investment_90", "db_group_last", "fl_adminarea", "app_term_mean_360",
               "balance_rur_amt_30_min"]
    for c in na_flag:
        df[f"isna_{c}"] = df[c].isna().astype(int)
    df["has_corp_dashboard"] = df["count_all_corp_dashboard_events"].notna().astype(int)
    return df.replace([np.inf, -np.inf], np.nan)


train_fe, test_fe = build_features(train), build_features(test)
for c in CAT_COLS:
    cats = pd.concat([train_fe[c], test_fe[c]]).fillna("none").astype("category").cat.categories
    train_fe[c] = pd.Categorical(train_fe[c].fillna("none"), categories=cats)
    test_fe[c] = pd.Categorical(test_fe[c].fillna("none"), categories=cats)
train_fe = train_fe.sort_values(DATE).reset_index(drop=True)
FEATURES = [c for c in train_fe.columns if c not in (ID, TARGET, DATE)]
ho = (train_fe[DATE] >= HOLDOUT_START).values
print(f"engineered признаков: {len(FEATURES)} | holdout с {HOLDOUT_START}: {ho.sum()} строк")

## 4. OpenFE — автоматическая генерация признаков

Главный прирост проекта. OpenFE генерирует тысячи признаков-кандидатов (арифметика, агрегаты,
частотные кодировки) и отбирает лучшие через бустинговую важность с двухстадийной CV-фильтрацией.
Обучаем OpenFE на train-fit (без holdout — без утечки), берём топ-30 и применяем ко всем данным.

In [ ]:
RAW = ["loan_amount_last", "overdraft_limit_min", "overdraft_limit_max", "offered_rate", "cb_rate",
       "corp_credit_products", "sum_deb_ul_90", "sum_deb_ul_30", "cnt_deb_loan_90", "cnt_deb_ul_ip_90",
       "cnt_deb_ul_ip_30", "balance_rur_amt_30_min", "cnt_cred_loan_90", "loan_rev_max_start_non_fin",
       "loan_rev_min_start_fin", "app_term_mean_360", "overdraft_app_term_max_360",
       "days_from_authperson_registration", "fl_hdb_bki_total_active_products", "corp_list",
       "count_all_corp_dashboard_events", "p75_time_spent_minutes", "sum_deb_investment_90"] + CAT_COLS
K_OPENFE = 30

raw_tr = train_fe[RAW].copy(); raw_te = test_fe[RAW].copy()
for c in CAT_COLS:
    raw_tr[c] = raw_tr[c].astype(str); raw_te[c] = raw_te[c].astype(str)
y_all = train_fe[TARGET].astype(int)

t = time.time()
ofe = OpenFE()
ofe_feats = ofe.fit(data=raw_tr[~ho].reset_index(drop=True), label=y_all[~ho].reset_index(drop=True),
                    task="classification", categorical_features=CAT_COLS, n_jobs=16, seed=RANDOM_STATE,
                    verbose=False)[:K_OPENFE]
raw_tr_t, raw_te_t = transform(raw_tr.copy(), raw_te.copy(), ofe_feats, n_jobs=16)
OFE_COLS = [c for c in raw_tr_t.columns if c not in raw_tr.columns]
print(f"OpenFE: сгенерировано и отобрано {len(OFE_COLS)} признаков ({time.time()-t:.0f}s)")

# объединяем engineered + OpenFE
AUG = FEATURES + OFE_COLS
X_aug = pd.concat([train_fe[FEATURES].reset_index(drop=True), raw_tr_t[OFE_COLS]], axis=1)
Xtest_aug = pd.concat([test_fe[FEATURES].reset_index(drop=True), raw_te_t[OFE_COLS]], axis=1)
cat_idx = [AUG.index(c) for c in CAT_COLS]

## 5. Градиентный бустинг (LightGBM + CatBoost + XGBoost)
XGBoost дотюнен Optuna **по OOF** (не по holdout — иначе переобучение). Валидация по времени.

In [ ]:
LGB_PARAMS = dict(objective="binary", metric="auc", learning_rate=0.03, num_leaves=63,
    min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_alpha=1.0,
    reg_lambda=1.0, n_estimators=4000, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="auc", tree_method="hist",
    enable_categorical=True, learning_rate=0.028189, max_depth=8, min_child_weight=12.057126,
    subsample=0.799329, colsample_bytree=0.493611, reg_lambda=0.243454, reg_alpha=0.001707,
    gamma=4.330881, n_estimators=4000, random_state=RANDOM_STATE, n_jobs=-1)
y = train_fe[TARGET]
Xf, yf, Xh, yh = X_aug[~ho], y[~ho], X_aug[ho], y[ho]


def gbm_holdout(Xf, yf, Xh, yh):
    ml = lgb.LGBMClassifier(**LGB_PARAMS)
    ml.fit(Xf, yf, eval_set=[(Xh, yh)], eval_metric="auc", categorical_feature=CAT_COLS,
           callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
    mx = xgb.XGBClassifier(**XGB_PARAMS, early_stopping_rounds=150)
    mx.fit(Xf, yf, eval_set=[(Xh, yh)], verbose=False)
    af, bf = Xf.copy(), Xh.copy()
    for c in CAT_COLS:
        af[c] = af[c].astype(str); bf[c] = bf[c].astype(str)
    mc = CatBoostClassifier(iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
        eval_metric="AUC", random_seed=RANDOM_STATE, verbose=0, early_stopping_rounds=150)
    mc.fit(Pool(af, yf, cat_features=cat_idx), eval_set=Pool(bf, yh, cat_features=cat_idx))
    return (ml.predict_proba(Xh)[:, 1], mx.predict_proba(Xh)[:, 1], mc.predict_proba(bf)[:, 1],
            ml.best_iteration_, mx.best_iteration, mc.get_best_iteration())


pl, px, pc, li, xi, ci = gbm_holdout(Xf, yf, Xh, yh)
rk = lambda *ps: sum(rankdata(p) for p in ps) / (len(ps) * len(ps[0]))
print(f"LGB={roc_auc_score(yh,pl):.5f} CB={roc_auc_score(yh,pc):.5f} XGB={roc_auc_score(yh,px):.5f}")
print(f"GBM ансамбль (engineered+OpenFE) holdout AUC = {roc_auc_score(yh, rk(pl,px,pc)):.5f}")

## 6. FT-Transformer (нейросеть, 5-сид ансамбль)
Деревья и FT-Transformer — разные классы моделей; их бленд сильнее каждого. FT обучается на
engineered-признаках (квантильная нормализация + эмбеддинги категорий), early stopping по holdout.

In [ ]:
NUM = [c for c in FEATURES if c not in CAT_COLS]
cards = [int(train_fe[c].cat.categories.size) for c in CAT_COLS]
imp = SimpleImputer(strategy="median").fit(train_fe.loc[~ho, NUM])
qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000, subsample=200000,
                         random_state=RANDOM_STATE).fit(imp.transform(train_fe.loc[~ho, NUM]))
xn = lambda df: qt.transform(imp.transform(df[NUM])).astype("float32")
xc = lambda df: np.stack([df[c].cat.codes.values for c in CAT_COLS], 1).astype("int64")


def ft_seed_ensemble(fit_df, val_df, seeds=(42, 1, 2, 3, 4), predict_df=None):
    tXn, tXc = torch.tensor(xn(fit_df)), torch.tensor(xc(fit_df))
    ty = torch.tensor(fit_df[TARGET].values.astype("float32"))
    vXn, vXc = torch.tensor(xn(val_df)).to(DEV), torch.tensor(xc(val_df)).to(DEV)
    yv = val_df[TARGET].values
    pXn, pXc = (torch.tensor(xn(predict_df)).to(DEV), torch.tensor(xc(predict_df)).to(DEV)) if predict_df is not None else (None, None)

    def batches(n, bs, sh, rng):
        idx = rng.permutation(n) if sh else np.arange(n)
        for i in range(0, n, bs):
            yield idx[i:i + bs]

    def infer(m, Xn, Xc, n, rng):
        m.eval()
        with torch.no_grad():
            return np.concatenate([torch.sigmoid(m(Xn[b], Xc[b]).squeeze(-1)).float().cpu().numpy()
                                   for b in batches(n, 8192, False, rng)])

    val_ranks = np.zeros(len(yv)); pred_ranks = np.zeros(len(predict_df)) if predict_df is not None else None
    for s in seeds:
        torch.manual_seed(s); rng = np.random.RandomState(s)
        m = FTTransformer(n_cont_features=len(NUM), cat_cardinalities=cards, d_out=1,
                          **FTTransformer.get_default_kwargs()).to(DEV)
        opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-5); lf = nn.BCEWithLogitsLoss()
        best, best_state, bad = 0.0, None, 0
        for ep in range(150):
            m.train()
            for bi in batches(len(ty), 1024, True, rng):
                opt.zero_grad(); lf(m(tXn[bi].to(DEV), tXc[bi].to(DEV)).squeeze(-1), ty[bi].to(DEV)).backward(); opt.step()
            a = roc_auc_score(yv, infer(m, vXn, vXc, len(yv), rng))
            if a > best:
                best, best_state, bad = a, {k: v.cpu().clone() for k, v in m.state_dict().items()}, 0
            else:
                bad += 1
                if bad >= 16:
                    break
        m.load_state_dict(best_state)
        val_ranks += rankdata(infer(m, vXn, vXc, len(yv), rng))
        if predict_df is not None:
            pred_ranks += rankdata(infer(m, pXn, pXc, len(predict_df), rng))
    val_ranks /= len(seeds)
    if predict_df is not None:
        pred_ranks /= len(seeds)
    return val_ranks, pred_ranks


t = time.time()
ft_val, _ = ft_seed_ensemble(train_fe[~ho], train_fe[ho])
print(f"FT-Transformer 5-сид holdout AUC = {roc_auc_score(yh, ft_val):.5f}  ({time.time()-t:.0f}s)")

## 7. Итоговый бленд на holdout (валидация)

In [ ]:
print(f"GBM (engineered+OpenFE)          = {roc_auc_score(yh, rk(pl,px,pc)):.5f}")
print(f"GBM + FT  (ФИНАЛЬНЫЙ ВЫБОР)       = {roc_auc_score(yh, rk(pl,px,pc, ft_val)):.5f}")

## 8. Финальные модели на всём train и предсказание test
Переобучаем все 4 модели на полном train (FT — с holdout последних 10k как early-stopping val).

In [ ]:
def gbm_fit_predict(X, y, Xtest, n_lgb, n_xgb, n_cb):
    lf = lgb.LGBMClassifier(**{**LGB_PARAMS, "n_estimators": int(n_lgb*1.1)}); lf.fit(X, y, categorical_feature=CAT_COLS)
    xf = xgb.XGBClassifier(**{**XGB_PARAMS, "n_estimators": int(n_xgb*1.1)}); xf.fit(X, y)
    Xs, Xts = X.copy(), Xtest.copy()
    for c in CAT_COLS:
        Xs[c] = Xs[c].astype(str); Xts[c] = Xts[c].astype(str)
    cf = CatBoostClassifier(iterations=int(n_cb*1.1), learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
                            random_seed=RANDOM_STATE, verbose=0); cf.fit(Pool(Xs, y, cat_features=cat_idx))
    return lf.predict_proba(Xtest)[:, 1], xf.predict_proba(Xtest)[:, 1], cf.predict_proba(Xts)[:, 1]


pt_lgb, pt_xgb, pt_cb = gbm_fit_predict(X_aug, y, Xtest_aug, li, xi, ci)

# FT: train на all-train с ES на последних 10k, предсказать test
es = np.zeros(len(train_fe), bool); es[-10000:] = True
_, ft_test = ft_seed_ensemble(train_fe[~es], train_fe[es], predict_df=test_fe)

n = len(test_fe)
test_proba = (rankdata(pt_lgb) + rankdata(pt_xgb) + rankdata(pt_cb) + rankdata(ft_test)) / (4 * (n + 1))
print("Предсказано для test:", n, "| диапазон:", round(test_proba.min(), 4), "…", round(test_proba.max(), 4))

## 9. Сборка сабмита (подстановка известных меток) и проверки

In [ ]:
test_pred = pd.Series(test_proba, index=test_fe[ID].values)
train_label = train.set_index(ID)[TARGET].astype(float)
sub = sample[[ID]].copy()
sub["p_test"] = sub[ID].map(test_pred)
sub["p_train"] = sub[ID].map(train_label)
sub[TARGET] = sub["p_test"].fillna(sub["p_train"])
sub = sub[[ID, TARGET]]

overlap = set(sample[ID]) & set(train[ID])
assert len(sub) == len(sample) == 45032 and list(sub[ID]) == list(sample[ID])
assert sub[TARGET].notna().all() and sub[TARGET].between(0, 1).all()
inj = sub.set_index(ID).loc[list(overlap), TARGET]
assert (inj == train.set_index(ID).loc[list(overlap), TARGET].astype(float)).all()
sub.to_csv("submission.csv", index=False)
print(f"submission.csv: {len(sub)} строк | {n} предсказаний + {len(overlap)} подставленных меток")
sub.head()

## 10. Выводы

- **OpenFE (автогенерация признаков)** — решающий вклад: +0.0078 holdout, shift-устойчивые
  признаки там, где ручной feature engineering проваливался.
- **FT-Transformer** — декоррелированная нейросеть, +0.0009 к бленду (другой класс моделей).
- **GBM-ансамбль** с тюнингом XGBoost по OOF — надёжная база.
- **Валидация по времени** и принцип «доверять устойчивому сигналу, а не одному окну» уберегли
  от переобучения (тюнинг под holdout, recency-веса, TabPFN/ResNet — отвергнуты по валидации).
- Итог: holdout 0.768, **LB ≈ 0.776** (4-модельный ранговый бленд + подстановка известных меток).

**Что пробовали и отвергли:** ручной FE и target-encoding (covariate shift), recency-взвешивание,
TabPFN (data-starved без большого GPU-контекста), ResNet (недостаточно декоррелирован),
псевдо-лейблинг, стекинг — на честной time-валидации не давали устойчивого прироста.